<a href="https://colab.research.google.com/github/abegithub2024/abegithub2024/blob/main/RUSLE_RF_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

DATA MANIPULATION & FORMAT FOR SOIL LOSS ESTIMATION
========================================================
Comprehensive guide for preparing soil erosion data for Random Forest modeling



In [ ]:
#DATA MANIPULATION & FORMAT TIPS FOR SOIL LOSS ESTIMATION
========================================================
#Comprehensive guide for preparing soil erosion data for Random Forest modeling

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder, QuantileTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. DATA LOADING & INITIAL INSPECTION
# ============================================================

def load_and_inspect_data(file_path):
    """
    Load data and perform initial quality check.
    Supports CSV, Excel, Shapefile (via geopandas), and JSON.
    """
    # Detect file type and load
    if file_path.endswith('.csv'):
        df = pd.read_csv(file_path)
    elif file_path.endswith(('.xlsx', '.xls')):
        df = pd.read_excel(file_path)
    elif file_path.endswith('.shp'):
        import geopandas as gpd
        gdf = gpd.read_file(file_path)
        df = pd.DataFrame(gdf.drop('geometry', axis=1))
        df['geometry'] = gdf.geometry  # Keep geometry separately if needed
    elif file_path.endswith('.json'):
        df = pd.read_json(file_path)
    else:
        raise ValueError("Unsupported file format")

    # Initial inspection
    print("=" * 60)
    print("DATA INSPECTION REPORT")
    print("=" * 60)
    print(f"\nShape: {df.shape}")
    print(f"\nColumn Types:\n{df.dtypes}")
    print(f"\nMissing Values:\n{df.isnull().sum()}")
    print(f"\nDuplicated Rows: {df.duplicated().sum()}")
    print(f"\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

    return df

# ============================================================
# 2. HANDLING MISSING VALUES (CRITICAL FOR RUSLE FACTORS)
# ============================================================

def handle_missing_values(df, numeric_strategy='auto', categorical_strategy='mode'):
    """
    Advanced missing value handling specific to soil erosion data.

    Strategies:
    - R, K factors: Median imputation (robust to outliers)
    - LS factor: KNN imputation (spatial context matters)
    - C factor: Mode or vegetation-based imputation
    - P factor: Default to 1.0 (no conservation practice)
    """
    df_clean = df.copy()

    # Identify column types
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df_clean.select_dtypes(include=['object', 'category']).columns.tolist()

    # Handle missing value codes (common in field data)
    missing_codes = [-999, -9999, 999, 9999, -1, 'NA', 'N/A', '', ' ']
    for code in missing_codes:
        df_clean = df_clean.replace(code, np.nan)

    print(f"\nMissing values after code conversion:\n{df_clean.isnull().sum()}")

    # Numeric imputation strategies
    if numeric_strategy == 'auto':
        # RUSLE-specific logic
        rusle_factors = ['R_factor', 'K_factor', 'LS_factor', 'C_factor', 'P_factor']

        for col in numeric_cols:
            if col in rusle_factors:
                if col in ['R_factor', 'K_factor']:
                    # Use median for rainfall and soil (stable across region)
                    imputer = SimpleImputer(strategy='median')
                elif col == 'LS_factor':
                    # Use KNN for topography (spatially correlated)
                    imputer = KNNImputer(n_neighbors=5)
                elif col == 'C_factor':
                    # Use vegetation-based imputation if NDVI available
                    if 'ndvi' in df_clean.columns:
                        df_clean[col] = df_clean.groupby(pd.cut(df_clean['ndvi'], 5))[col].transform(
                            lambda x: x.fillna(x.median()))
                        continue
                    else:
                        imputer = SimpleImputer(strategy='median')
                elif col == 'P_factor':
                    # Default to 1.0 (no protection)
                    df_clean[col] = df_clean[col].fillna(1.0)
                    continue

                df_clean[[col]] = imputer.fit_transform(df_clean[[col]])
            else:
                # Other numeric: median
                df_clean[[col]] = SimpleImputer(strategy='median').fit_transform(df_clean[[col]])

    # Categorical imputation
    if categorical_strategy == 'mode':
        for col in categorical_cols:
            if df_clean[col].isnull().sum() > 0:
                mode_val = df_clean[col].mode()[0] if not df_clean[col].mode().empty else 'unknown'
                df_clean[col] = df_clean[col].fillna(mode_val)

    return df_clean

# ============================================================
# 3. STANDARDIZING CATEGORICAL VARIABLES
# ============================================================

def standardize_categories(df, column_mappings=None):
    """
    Standardize text data: clean whitespace, lowercase, map synonyms.

    Common soil erosion categories:
    - Land cover types
    - Soil texture classes
    - Conservation practice types
    """
    df_std = df.copy()

    # Default mappings for common fields
    default_mappings = {
        'land_cover': {
            'forest': ['forest', 'forests', 'woodland', 'woods', 'tree', 'trees'],
            'agriculture': ['agriculture', 'agricultural', 'crop', 'crops', 'cultivated', 'farm', 'farming', 'ag'],
            'grassland': ['grassland', 'grass', 'pasture', 'meadow', 'range', 'rangeland'],
            'bare': ['bare', 'barren', 'exposed', 'denuded', 'sparse'],
            'urban': ['urban', 'built', 'construction', 'residential', 'commercial', 'industrial'],
            'water': ['water', 'wetland', 'wetlands', 'marsh', 'swamp', 'river', 'lake']
        },
        'soil_texture': {
            'sand': ['sand', 'sandy', 'coarse'],
            'loam': ['loam', 'loamy', 'medium'],
            'clay': ['clay', 'clayey', 'fine', 'heavy']
        }
    }

    # Apply standardization
    for col in df_std.select_dtypes(include=['object']).columns:
        # Clean text
        df_std[col] = df_std[col].astype(str).str.strip().str.lower()

        # Apply mappings if column exists in mappings
        mappings = column_mappings or default_mappings
        if col in mappings:
            reverse_map = {}
            for std_val, variants in mappings[col].items():
                for variant in variants:
                    reverse_map[variant] = std_val

            df_std[col] = df_std[col].map(reverse_map).fillna(df_std[col])

    return df_std

# ============================================================
# 4. OUTLIER DETECTION & TREATMENT
# ============================================================

def handle_outliers(df, method='iqr', treatment='cap', columns=None):
    """
    Detect and treat outliers in soil erosion data.

    Methods:
    - iqr: Interquartile range (1.5 * IQR rule)
    - zscore: Z-score > 3
    - isolation_forest: ML-based outlier detection

    Treatment:
    - cap: Winsorization (cap at percentiles)
    - remove: Remove outlier rows
    - transform: Log transformation
    """
    df_clean = df.copy()
    numeric_cols = columns or df.select_dtypes(include=[np.number]).columns.tolist()

    outlier_report = {}

    for col in numeric_cols:
        if col in ['plot_id', 'sample_id', 'latitude', 'longitude']:
            continue

        # Detection
        if method == 'iqr':
            Q1 = df_clean[col].quantile(0.25)
            Q3 = df_clean[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            outliers = df_clean[(df_clean[col] < lower) | (df_clean[col] > upper)]

        elif method == 'zscore':
            z_scores = np.abs(stats.zscore(df_clean[col].dropna()))
            outliers = df_clean[z_scores > 3]
            lower, upper = None, None

        outlier_report[col] = len(outliers)

        # Treatment
        if treatment == 'cap':
            # Physical limits for soil erosion factors
            limits = {
                'R_factor': (0, 2000),
                'K_factor': (0, 1),
                'LS_factor': (0, 100),
                'C_factor': (0, 1),
                'P_factor': (0, 1),
                'slope_deg': (0, 90),
                'elevation': (-500, 9000),
                'soil_loss': (0, 1000)  # Extreme upper limit
            }

            if col in limits:
                low, high = limits[col]
            else:
                low, high = df_clean[col].quantile(0.01), df_clean[col].quantile(0.99)

            df_clean[col] = df_clean[col].clip(lower=low, upper=high)

        elif treatment == 'remove' and len(outliers) > 0:
            df_clean = df_clean[~df_clean.index.isin(outliers.index)]

        elif treatment == 'transform':
            # Log transform for skewed data (soil loss, LS factor)
            if df_clean[col].min() >= 0:
                df_clean[f'{col}_log'] = np.log1p(df_clean[col])

    print(f"\nOutlier report ({method} method):")
    for col, count in outlier_report.items():
        if count > 0:
            print(f"  {col}: {count} outliers detected")

    return df_clean

# ============================================================
# 5. FEATURE ENGINEERING FOR SOIL EROSION
# ============================================================

def engineer_features(df):
    """
    Create derived features relevant to soil loss modeling.
    These capture physical relationships better than raw data.
    """
    df_feat = df.copy()

    # 1. RUSLE Interaction Terms (capture multiplicative effects)
    if all(f in df_feat.columns for f in ['R_factor', 'K_factor', 'LS_factor']):
        df_feat['RK_interaction'] = df_feat['R_factor'] * df_feat['K_factor']
        df_feat['RLS_interaction'] = df_feat['R_factor'] * df_feat['LS_factor']

    # 2. Topographic indices
    if 'slope_deg' in df_feat.columns:
        # Slope in radians for calculations
        df_feat['slope_rad'] = np.radians(df_feat['slope_deg'])
        df_feat['slope_sin'] = np.sin(df_feat['slope_rad'])
        df_feat['slope_tan'] = np.tan(df_feat['slope_rad'])

        # Slope categories (important for erosion regimes)
        df_feat['slope_class'] = pd.cut(
            df_feat['slope_deg'],
            bins=[0, 5, 15, 30, 90],
            labels=['gentle', 'moderate', 'steep', 'very_steep']
        )

    if all(c in df_feat.columns for c in ['elevation', 'slope_deg']):
        # Elevation-Slope interaction (mountainous erosion)
        df_feat['elev_slope'] = df_feat['elevation'] * df_feat['slope_deg'] / 1000

    # 3. Hydrological features
    if 'drainage_density' in df_feat.columns and 'slope_deg' in df_feat.columns:
        # Topographic Wetness Index proxy
        df_feat['twi_proxy'] = np.log(
            (df_feat['drainage_density'] + 0.001) / (np.tan(np.radians(df_feat['slope_deg'])) + 0.001)
        )

    if 'distance_to_river' in df_feat.columns:
        # Proximity categories
        df_feat['river_proximity'] = pd.cut(
            df_feat['distance_to_river'],
            bins=[0, 100, 500, 1000, float('inf')],
            labels=['very_close', 'close', 'moderate', 'far']
        )
        # Inverse distance (stronger effect when close)
        df_feat['inv_dist_river'] = 1 / (df_feat['distance_to_river'] + 1)

    # 4. Vegetation features
    if 'ndvi' in df_feat.columns:
        # NDVI categories
        df_feat['ndvi_class'] = pd.cut(
            df_feat['ndvi'],
            bins=[-1, 0, 0.2, 0.5, 1],
            labels=['water/shadow', 'sparse', 'moderate', 'dense']
        )
        # Seasonal adjustment proxy
        if 'date' in df_feat.columns:
            df_feat['month'] = pd.to_datetime(df_feat['date']).dt.month
            df_feat['season'] = df_feat['month'].map({
                12: 'winter', 1: 'winter', 2: 'winter',
                3: 'spring', 4: 'spring', 5: 'spring',
                6: 'summer', 7: 'summer', 8: 'summer',
                9: 'autumn', 10: 'autumn', 11: 'autumn'
            })

    # 5. Erosion potential index (simplified RUSLE without C and P)
    rusle_cols = ['R_factor', 'K_factor', 'LS_factor']
    if all(c in df_feat.columns for c in rusle_cols):
        df_feat['erosion_potential'] = (
            df_feat['R_factor'] * df_feat['K_factor'] * df_feat['LS_factor']
        )

    # 6. Conservation effectiveness
    if all(c in df_feat.columns for c in ['C_factor', 'P_factor']):
        df_feat['management_effectiveness'] = (1 - df_feat['C_factor']) * (1 - df_feat['P_factor'])

    return df_feat

# ============================================================
# 6. DATA TRANSFORMATION & SCALING
# ============================================================

def transform_features(df, numeric_cols=None, scaler_type='robust'):
    """
    Apply appropriate transformations for soil erosion data.

    scaler_type options:
    - robust: RobustScaler (handles outliers, best for erosion data)
    - standard: StandardScaler (zero mean, unit variance)
    - quantile: QuantileTransformer (normalizes distributions)
    - log: Log transformation (for skewed data like soil loss)
    """
    df_trans = df.copy()
    numeric_cols = numeric_cols or df.select_dtypes(include=[np.number]).columns.tolist()

    # Exclude ID columns and target
    exclude_cols = ['plot_id', 'sample_id', 'latitude', 'longitude', 'soil_loss', 'soil_loss_actual']
    feature_cols = [c for c in numeric_cols if c not in exclude_cols]

    # Distribution transformations
    skewed_cols = []
    for col in feature_cols:
        if df_trans[col].min() >= 0 and df_trans[col].skew() > 1:
            skewed_cols.append(col)
            df_trans[f'{col}_log'] = np.log1p(df_trans[col])

    print(f"Log-transformed skewed columns: {skewed_cols}")

    # Scaling
    if scaler_type == 'robust':
        scaler = RobustScaler()
    elif scaler_type == 'standard':
        scaler = StandardScaler()
    elif scaler_type == 'quantile':
        scaler = QuantileTransformer(output_distribution='normal')
    else:
        return df_trans

    # Scale features (not target)
    scale_cols = [c for c in feature_cols if c not in skewed_cols]
    if scale_cols:
        df_trans[[f'{c}_scaled' for c in scale_cols]] = scaler.fit_transform(df_trans[scale_cols])

    return df_trans, scaler

# ============================================================
# 7. SPATIAL DATA HANDLING
# ============================================================

def prepare_spatial_data(df, lat_col='latitude', lon_col='longitude', crs='EPSG:4326'):
    """
    Prepare spatial data for GIS integration.
    Creates GeoDataFrame and extracts spatial features.
    """
    try:
        import geopandas as gpd
        from shapely.geometry import Point

        # Create geometry
        geometry = [Point(xy) for xy in zip(df[lon_col], df[lat_col])]
        gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=crs)

        # Extract spatial features
        gdf['x_coord'] = gdf.geometry.x
        gdf['y_coord'] = gdf.geometry.y

        # If in projected coordinates, calculate distances
        if crs != 'EPSG:4326':
            gdf['area_km2'] = gdf.geometry.area / 1e6

        return gdf

    except ImportError:
        print("geopandas not installed. Returning DataFrame with coordinates.")
        df['x_coord'] = df[lon_col]
        df['y_coord'] = df[lat_col]
        return df

# ============================================================
# 8. FINAL DATA FORMATTING FOR ML
# ============================================================

def format_for_ml(df, target_col='soil_loss', feature_cols=None,
                  encode_categorical=True, drop_original_cat=True):
    """
    Final formatting: encode categoricals, select features, ensure correct types.
    Returns X, y ready for sklearn.
    """
    df_ml = df.copy()

    # Separate target
    y = df_ml[target_col]
    X = df_ml.drop(columns=[target_col])

    # Select features if specified
    if feature_cols:
        X = X[[c for c in feature_cols if c in X.columns]]

    # Encode categoricals
    if encode_categorical:
        cat_cols = X.select_dtypes(include=['object', 'category']).columns
        le_dict = {}

        for col in cat_cols:
            le = LabelEncoder()
            X[f'{col}_encoded'] = le.fit_transform(X[col].astype(str))
            le_dict[col] = le

            if drop_original_cat:
                X = X.drop(columns=[col])

    # Ensure all numeric
    X = X.select_dtypes(include=[np.number])

    # Remove any remaining NaN
    X = X.fillna(X.median())

    print(f"\nFinal ML format:")
    print(f"  X shape: {X.shape}")
    print(f"  y shape: {y.shape}")
    print(f"  Features: {list(X.columns)}")

    return X, y

# ============================================================
# 9. COMPLETE PIPELINE EXAMPLE
# ============================================================

def full_preprocessing_pipeline(file_path):
    """
    Complete preprocessing pipeline for soil loss data.
    """
    # 1. Load
    df = load_and_inspect_data(file_path)

    # 2. Clean missing values
    df = handle_missing_values(df)

    # 3. Standardize text
    df = standardize_categories(df)

    # 4. Handle outliers
    df = handle_outliers(df, method='iqr', treatment='cap')

    # 5. Engineer features
    df = engineer_features(df)

    # 6. Transform
    df, scaler = transform_features(df, scaler_type='robust')

    # 7. Format for ML
    X, y = format_for_ml(df, target_col='soil_loss')

    return X, y, df

# ============================================================
# 10. DATA VALIDATION CHECKS
# ============================================================

def validate_soil_data(df):
    """
    Run validation checks specific to soil erosion data.
    """
    errors = []
    warnings_list = []

    # Check RUSLE factor ranges
    rusle_limits = {
        'R_factor': (0, 2000),
        'K_factor': (0, 1),
        'LS_factor': (0, 100),
        'C_factor': (0, 1),
        'P_factor': (0, 1)
    }

    for factor, (min_val, max_val) in rusle_limits.items():
        if factor in df.columns:
            if df[factor].min() < min_val or df[factor].max() > max_val:
                errors.append(f"{factor} outside valid range [{min_val}, {max_val}]")

    # Check for negative soil loss
    if 'soil_loss' in df.columns and (df['soil_loss'] < 0).any():
        errors.append("Negative soil loss values detected")

    # Check coordinate validity
    if 'latitude' in df.columns:
        if df['latitude'].abs().max() > 90:
            errors.append("Invalid latitude values (>90)")
    if 'longitude' in df.columns:
        if df['longitude'].abs().max() > 180:
            errors.append("Invalid longitude values (>180)")

    # Warnings for suspicious data
    if 'soil_loss' in df.columns:
        if df['soil_loss'].max() > 1000:
            warnings_list.append("Very high soil loss values (>1000) - check units")
        if (df['soil_loss'] == 0).sum() / len(df) > 0.5:
            warnings_list.append("More than 50% zero soil loss values")

    print("\n" + "=" * 60)
    print("VALIDATION REPORT")
    print("=" * 60)
    if errors:
        print(f"\n❌ ERRORS ({len(errors)}):")
        for e in errors:
            print(f"  - {e}")
    else:
        print("\n✅ No critical errors found")

    if warnings_list:
        print(f"\n⚠️  WARNINGS ({len(warnings_list)}):")
        for w in warnings_list:
            print(f"  - {w}")

    return len(errors) == 0
